# 特徴量エンジニアリング × 全モデル検証

**目的**: 04_baseline_lgbm_feature_engineeringで実装した特徴量エンジニアリングを全7モデルで検証し、最適なモデルを選定する

**モデル一覧**:
1. LightGBM
2. XGBoost
3. CatBoost
4. HistGradientBoosting
5. Logistic Regression
6. Neural Network
7. TabNet

**CV戦略**: TimeSeriesSplit（cv_strategy="timeseries"）

**出力**: 各モデルのOOF予測をCSVファイルに保存

```markdown
## システムメモリの確認
Colab Proではより多くのメモリが提供されます。以下のコードで現在のシステムメモリの使用状況と合計メモリ量を確認できます。
```

In [1]:
import psutil

# メモリ情報を取得
mem = psutil.virtual_memory()
total_memory_gb = mem.total / (1024**3) # バイトをギガバイトに変換
available_memory_gb = mem.available / (1024**3)
used_memory_gb = mem.used / (1024**3)

print(f"総メモリ: {total_memory_gb:.2f} GB")
print(f"利用可能メモリ: {available_memory_gb:.2f} GB")
print(f"使用済みメモリ: {used_memory_gb:.2f} GB")

# より詳細なメモリ情報 (linux コマンド)
!cat /proc/meminfo

総メモリ: 172.95 GB
利用可能メモリ: 169.61 GB
使用済みメモリ: 1.77 GB
MemTotal:       181349372 kB
MemFree:        166862112 kB
MemAvailable:   177853088 kB
Buffers:          552344 kB
Cached:         11413516 kB
SwapCached:            0 kB
Active:          1168712 kB
Inactive:       12167988 kB
Active(anon):       2412 kB
Inactive(anon):  1371876 kB
Active(file):    1166300 kB
Inactive(file): 10796112 kB
Unevictable:          12 kB
Mlocked:              12 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:             54200 kB
Writeback:             0 kB
AnonPages:       1369500 kB
Mapped:           664420 kB
Shmem:              3392 kB
KReclaimable:     668588 kB
Slab:             856928 kB
SReclaimable:     668588 kB
SUnreclaim:       188340 kB
KernelStack:       23376 kB
PageTables:        16856 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:    90674684 kB
Committed_AS:   11838464 kB
VmallocTotal:   34359

In [2]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# プロジェクトルートの設定（Google Drive）
PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026") # Google Drive内の正しいパスに修正
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [3]:


import datetime
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import LabelEncoder

from common.utils.logger import get_logger

warnings.filterwarnings("ignore")

# 基本パラメータ
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"
DATA_DIR = PROJECT_ROOT / "data" / "input"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"TARGET_COL: {TARGET_COL}")
print(f"ID_COL: {ID_COL}")

PROJECT_ROOT: /content/drive/MyDrive/jaggle_2026
DATA_DIR: /content/drive/MyDrive/jaggle_2026/data/input
TARGET_COL: 10年定着ラベル
ID_COL: 社員ID


## 1. 特徴量エンジニアリング関数定義

04_baseline_lgbm_feature_engineeringから流用した特徴量生成関数群

In [4]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """
    月次データから集約特徴量を生成
    """
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []

    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)

        features = {}
        features["社員ID"] = employee_id

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue

            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            # 1. 統計量ベース
            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan

            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            # 2. 時期別統計量
            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]

            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()

            # 3. トレンド特徴量
            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan

                first_val = valid_values[0]
                last_val = valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = np.nan
                features[f"{col}_diff"] = np.nan
                features[f"{col}_ratio"] = np.nan

        features_list.append(features)

    return pd.DataFrame(features_list)

print("✅ 月次集約特徴量関数の定義完了")

✅ 月次集約特徴量関数の定義完了


In [5]:
def create_monthly_categorical_change_features(monthly_df, employee_ids):
    """
    月次カテゴリカル変数の変化パターン特徴量を生成
    """
    categorical_cols = [
        "部署ID", "職種", "役割", "等級", "勤務地", "上司ID"
    ]

    features_list = []

    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)

        features = {}
        features["社員ID"] = employee_id

        for col in categorical_cols:
            if col not in emp_data.columns:
                continue

            values = emp_data[col].values

            # 変更回数
            changes = 0
            for i in range(1, len(values)):
                if pd.notna(values[i]) and pd.notna(values[i-1]):
                    if values[i] != values[i-1]:
                        changes += 1
            features[f"{col}_changes"] = changes

            # ユニーク数
            unique_values = pd.Series(values).dropna().unique()
            features[f"{col}_unique_count"] = len(unique_values)

            # 初期と最終が同じか
            if len(values) >= 2:
                first_valid = pd.Series(values).dropna().iloc[0] if len(pd.Series(values).dropna()) > 0 else None
                last_valid = pd.Series(values).dropna().iloc[-1] if len(pd.Series(values).dropna()) > 0 else None
                features[f"{col}_same_start_end"] = int(first_valid == last_valid) if first_valid is not None and last_valid is not None else np.nan
            else:
                features[f"{col}_same_start_end"] = np.nan

            # 最初の変更が起きた月
            first_change_month = np.nan
            for i in range(1, len(values)):
                if pd.notna(values[i]) and pd.notna(values[i-1]):
                    if values[i] != values[i-1]:
                        first_change_month = emp_data.iloc[i]["経過月数"]
                        break
            features[f"{col}_first_change_month"] = first_change_month

        # 月末在籍状態の特徴量
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            # 休職経験フラグ
            features["leave_of_absence_flag"] = int("休職" in status_values)
            # 休職月数
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
            # 在籍継続月数
            consecutive_months = 0
            for status in status_values:
                if status == "在籍":
                    consecutive_months += 1
                else:
                    break
            features["consecutive_active_months"] = consecutive_months

        features_list.append(features)

    return pd.DataFrame(features_list)

print("✅ 月次カテゴリ変化特徴量関数の定義完了")

✅ 月次カテゴリ変化特徴量関数の定義完了


In [6]:
def create_missing_value_features(monthly_df, employee_ids):
    """欠損値パターン特徴量を生成"""
    missing_target_cols = [
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度",
        "顧客満足度評価", "担当プロジェクト数"
    ]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {}
        features["社員ID"] = employee_id
        for col in missing_target_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            total_months = len(values)
            missing_count = pd.isna(values).sum()
            features[f"{col}_missing_rate"] = missing_count / total_months if total_months > 0 else np.nan
            first_observed_month = np.nan
            for i, val in enumerate(values):
                if pd.notna(val):
                    first_observed_month = emp_data.iloc[i]["経過月数"]
                    break
            features[f"{col}_first_observed_month"] = first_observed_month
            max_streak = 0
            current_streak = 0
            for val in values:
                if pd.isna(val):
                    current_streak += 1
                    max_streak = max(max_streak, current_streak)
                else:
                    current_streak = 0
            features[f"{col}_max_missing_streak"] = max_streak
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 欠損値特徴量関数の定義完了")

✅ 欠損値特徴量関数の定義完了


In [7]:
def create_domain_knowledge_features(monthly_df, employee_ids):
    """ドメイン知識ベースの特徴量を生成"""
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {}
        features["社員ID"] = employee_id
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if eval_mean_list else np.nan
        if "有給取得日数" in emp_data.columns:
            features["paid_leave_rate"] = emp_data["有給取得日数"].mean() / 20
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
            if "研修時間" in emp_data.columns:
                training_mean = emp_data["研修時間"].mean()
                overtime_mean = emp_data["残業時間"].mean()
                features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        if "研修時間" in emp_data.columns:
            training_values = emp_data["研修時間"].values
            valid_training = training_values[~pd.isna(training_values)]
            if len(valid_training) >= 2:
                valid_indices = np.where(~pd.isna(training_values))[0]
                slope, _, _, _, _ = stats.linregress(valid_indices, valid_training)
                features["training_trend"] = slope
            else:
                features["training_trend"] = np.nan
        if "上司との面談実施回数" in emp_data.columns:
            features["meeting_frequency"] = emp_data["上司との面談実施回数"].mean()
        if "情報共有件数" in emp_data.columns:
            features["info_sharing_activity"] = emp_data["情報共有件数"].mean()
        if "残業時間" in emp_data.columns:
            features["overtime_stress"] = emp_data["残業時間"].mean()
        if "欠勤日数" in emp_data.columns:
            absence_values = emp_data["欠勤日数"].values
            valid_absence = absence_values[~pd.isna(absence_values)]
            if len(valid_absence) >= 2:
                valid_indices = np.where(~pd.isna(absence_values))[0]
                slope, _, _, _, _ = stats.linregress(valid_indices, valid_absence)
                features["absence_trend"] = slope
            else:
                features["absence_trend"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ ドメイン知識特徴量関数の定義完了")

✅ ドメイン知識特徴量関数の定義完了


## 2. 設定とデータ読み込み

In [8]:
SCRIPT_NAME = "07_poc_feature_engineering_for_models"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
SEED = 42

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"OUTPUT_DIR: {OUTPUT_DIR}")

[2026-08-06 16:38:30] [INFO] === [07_poc_feature_engineering_for_models] 実験開始 ===


INFO:07_poc_feature_engineering_for_models:=== [07_poc_feature_engineering_for_models] 実験開始 ===


[2026-08-06 16:38:31] [INFO] OUTPUT_DIR: /content/drive/MyDrive/jaggle_2026/data/output/20260806


INFO:07_poc_feature_engineering_for_models:OUTPUT_DIR: /content/drive/MyDrive/jaggle_2026/data/output/20260806


In [9]:
train_persona = pd.read_csv(DATA_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(DATA_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(DATA_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(DATA_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona: {train_persona.shape}, Test Persona: {test_persona.shape}")
logger.info(f"Train Monthly: {train_monthly.shape}, Test Monthly: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].unique()
test_ids = test_persona[ID_COL].unique()

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-06 16:38:36] [INFO] Train Persona: (2761, 20), Test Persona: (2502, 19)


INFO:07_poc_feature_engineering_for_models:Train Persona: (2761, 20), Test Persona: (2502, 19)


[2026-08-06 16:38:36] [INFO] Train Monthly: (65754, 29), Test Monthly: (60048, 29)


INFO:07_poc_feature_engineering_for_models:Train Monthly: (65754, 29), Test Monthly: (60048, 29)


[2026-08-06 16:38:36] [INFO] 定着率: 0.5647


INFO:07_poc_feature_engineering_for_models:定着率: 0.5647


[2026-08-06 16:38:36] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:07_poc_feature_engineering_for_models:Train IDs: 2761, Test IDs: 2502


## 3. 特徴量生成の実行

In [10]:
logger.info("-" * 60)
logger.info("特徴量生成開始")
logger.info("-" * 60)

logger.info("月次集約特徴量を生成中...")
train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)
logger.info(f"Train: {train_monthly_agg.shape}, Test: {test_monthly_agg.shape}")

logger.info("月次カテゴリ変化特徴量を生成中...")
train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)
logger.info(f"Train: {train_cat_change.shape}, Test: {test_cat_change.shape}")

logger.info("欠損値特徴量を生成中...")
train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)
logger.info(f"Train: {train_missing.shape}, Test: {test_missing.shape}")

logger.info("ドメイン知識特徴量を生成中...")
train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)
logger.info(f"Train: {train_domain.shape}, Test: {test_domain.shape}")

[2026-08-06 16:38:36] [INFO] ------------------------------------------------------------


INFO:07_poc_feature_engineering_for_models:------------------------------------------------------------


[2026-08-06 16:38:36] [INFO] 特徴量生成開始


INFO:07_poc_feature_engineering_for_models:特徴量生成開始


[2026-08-06 16:38:36] [INFO] ------------------------------------------------------------


...

INFO:07_poc_feature_engineering_for_models:Train: (2761, 22), Test: (2502, 22)


[2026-08-06 16:40:57] [INFO] ドメイン知識特徴量を生成中...


INFO:07_poc_feature_engineering_for_models:ドメイン知識特徴量を生成中...


[2026-08-06 16:41:19] [INFO] Train: (2761, 10), Test: (2502, 10)


INFO:07_poc_feature_engineering_for_models:Train: (2761, 10), Test: (2502, 10)


In [11]:
logger.info("時間的特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

logger.info("交互作用特徴量を生成中...")
train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]

logger.info("特徴量生成完了")

[2026-08-06 16:41:19] [INFO] 時間的特徴量を生成中...


INFO:07_poc_feature_engineering_for_models:時間的特徴量を生成中...


[2026-08-06 16:41:19] [INFO] 交互作用特徴量を生成中...


INFO:07_poc_feature_engineering_for_models:交互作用特徴量を生成中...


[2026-08-06 16:41:19] [INFO] 特徴量生成完了


INFO:07_poc_feature_engineering_for_models:特徴量生成完了


## 4. 特徴量の統合と前処理

In [12]:
logger.info("-" * 60)
logger.info("特徴量の統合")
logger.info("-" * 60)

train_persona_features = train_persona.drop(columns=[TARGET_COL])

train_features = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
train_features = train_features.merge(train_cat_change, on=ID_COL, how="left")
train_features = train_features.merge(train_missing, on=ID_COL, how="left")
train_features = train_features.merge(train_domain, on=ID_COL, how="left")

test_features = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
test_features = test_features.merge(test_cat_change, on=ID_COL, how="left")
test_features = test_features.merge(test_missing, on=ID_COL, how="left")
test_features = test_features.merge(test_domain, on=ID_COL, how="left")

logger.info(f"Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-06 16:41:19] [INFO] ------------------------------------------------------------


INFO:07_poc_feature_engineering_for_models:------------------------------------------------------------


[2026-08-06 16:41:19] [INFO] 特徴量の統合


INFO:07_poc_feature_engineering_for_models:特徴量の統合


[2026-08-06 16:41:19] [INFO] ------------------------------------------------------------


INFO:07_poc_feature_engineering_for_models:------------------------------------------------------------


[2026-08-06 16:41:19] [INFO] Train: (2761, 288), Test: (2502, 288)


INFO:07_poc_feature_engineering_for_models:Train: (2761, 288), Test: (2502, 288)


In [13]:
logger.info("カテゴリカル変数の処理...")
cat_cols = ["入社区分", "性別", "最終学歴", "専攻分野", "採用経路", "初期職種", "初期勤務地", "初期等級", "初期役割"]

for col in cat_cols:
    if col in train_features.columns:
        le = LabelEncoder()
        combined = pd.concat([train_features[col].fillna("missing"), test_features[col].fillna("missing")])
        le.fit(combined)
        train_features[col] = le.transform(train_features[col].fillna("missing"))
        test_features[col] = le.transform(test_features[col].fillna("missing"))

logger.info("不要な列の削除...")
drop_cols = ["社員ID", "入社日", "入社時メモ", "上司からのフィードバック", "同僚からのフィードバック", "初期部署ID", "前職職種"]
drop_cols_exist = [col for col in drop_cols if col in train_features.columns]
train_features = train_features.drop(columns=drop_cols_exist)
test_features = test_features.drop(columns=drop_cols_exist)

X_train = train_features.fillna(-999)
X_test = test_features.fillna(-999)

logger.info(f"X_train Shape: {X_train.shape}")
logger.info(f"X_test Shape: {X_test.shape}")
logger.info(f"y_train Shape: {y_train.shape}")
logger.info(f"最終的な特徴量数: {X_train.shape[1]}")

[2026-08-06 16:41:19] [INFO] カテゴリカル変数の処理...


INFO:07_poc_feature_engineering_for_models:カテゴリカル変数の処理...


[2026-08-06 16:41:19] [INFO] 不要な列の削除...


INFO:07_poc_feature_engineering_for_models:不要な列の削除...


[2026-08-06 16:41:19] [INFO] X_train Shape: (2761, 281)


...

INFO:07_poc_feature_engineering_for_models:X_test Shape: (2502, 281)


[2026-08-06 16:41:19] [INFO] y_train Shape: (2761,)


INFO:07_poc_feature_engineering_for_models:y_train Shape: (2761,)


[2026-08-06 16:41:19] [INFO] 最終的な特徴量数: 281


INFO:07_poc_feature_engineering_for_models:最終的な特徴量数: 281


## 5. モデル実行と結果保存

7つのモデルをTimeSeriesSplit (cv_strategy="timeseries")で実行し、各モデルのOOF予測をCSVファイルに保存します。

In [14]:
%pip install lightgbm xgboost catboost torch pytorch-tabnet --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 20.3 MB/s eta 0:00:00


In [15]:
base_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": None,
    "cv_strategy": "timeseries"
}

results = {}

logger.info("=" * 60)
logger.info("全モデルの実行開始")
logger.info("=" * 60)

[2026-08-06 16:41:33] [INFO] ============================================================


INFO:07_poc_feature_engineering_for_models:============================================================


[2026-08-06 16:41:33] [INFO] 全モデルの実行開始


INFO:07_poc_feature_engineering_for_models:全モデルの実行開始


[2026-08-06 16:41:33] [INFO] ============================================================


INFO:07_poc_feature_engineering_for_models:============================================================


In [16]:
print("=" * 60)
print(f"{datetime.datetime.now()} - LightGBM v2 実行開始")
print("=" * 60)

from common.lgbm.lgbm_model_v2 import run_lgb

lgbm_params = base_params.copy()
lgbm_params.update({
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "early_stopping_rounds": 50,
    "n_estimators": 1000,
})

# 入社日をTimeSeriesSplit用に追加
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

data = {"X_train": X_train_with_sort, "y_train": y_train, "X_test": X_test, "sort_col": "入社日"}
result_data, _ = run_lgb(data, lgbm_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nLightGBM v2 OOF Score (Log Loss): {oof_score:.6f}")
results["LightGBM"] = oof_score

# CSV保存
submission_lgbm = pd.DataFrame({
    ID_COL: test_persona[ID_COL],
    TARGET_COL: test_preds
})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_lgbm.csv"
submission_lgbm.to_csv(submission_path, index=False,header=False)
logger.info(f"LightGBM予測結果保存: {submission_path}")

2026-08-06 16:41:33.467608 - LightGBM v2 実行開始

LightGBM v2 OOF Score (Log Loss): 0.595333
[2026-08-06 16:41:38] [INFO] LightGBM予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_lgbm.csv


INFO:07_poc_feature_engineering_for_models:LightGBM予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_lgbm.csv


In [17]:
print("=" * 60)
print(f"{datetime.datetime.now()} - XGBoost v2 実行開始")
print("=" * 60)

from common.xgboost.xgb_model_v2 import run_xgb

xgboost_params = base_params.copy()
xgboost_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "early_stopping_rounds": 50,
    "n_estimators": 1000,
})

# 入社日をTimeSeriesSplit用に追加
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

data = {"X_train": X_train_with_sort, "y_train": y_train, "X_test": X_test, "sort_col": "入社日"}
result_data, _ = run_xgb(data, xgboost_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nXGBoost v2 OOF Score (Log Loss): {oof_score:.6f}")
results["XGBoost"] = oof_score

submission_xgb = pd.DataFrame({ID_COL: test_persona[ID_COL], TARGET_COL: test_preds})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_xgb.csv"
submission_xgb.to_csv(submission_path, index=False,header=False)
logger.info(f"XGBoost予測結果保存: {submission_path}")

2026-08-06 16:41:38.798568 - XGBoost v2 実行開始

XGBoost v2 OOF Score (Log Loss): 0.626103
[2026-08-06 16:41:41] [INFO] XGBoost予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_xgb.csv


INFO:07_poc_feature_engineering_for_models:XGBoost予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_xgb.csv


In [18]:
print("=" * 60)
print(f"{datetime.datetime.now()} - CatBoost v2 実行開始")
print("=" * 60)

from common.catboost.cat_model_v2 import run_cat

catboost_params = base_params.copy()
catboost_params.update({
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": 1000,
    "early_stopping_rounds": 50,
    "verbose": False,
})

# 入社日をTimeSeriesSplit用に追加
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

data = {"X_train": X_train_with_sort, "y_train": y_train, "X_test": X_test, "sort_col": "入社日"}
result_data, _ = run_cat(data, catboost_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nCatBoost v2 OOF Score (Log Loss): {oof_score:.6f}")
results["CatBoost"] = oof_score

submission_cat = pd.DataFrame({ID_COL: test_persona[ID_COL], TARGET_COL: test_preds})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_cat.csv"
submission_cat.to_csv(submission_path, index=False,header=False)
logger.info(f"CatBoost予測結果保存: {submission_path}")

2026-08-06 16:41:41.996279 - CatBoost v2 実行開始

CatBoost v2 OOF Score (Log Loss): 0.586837
[2026-08-06 16:41:49] [INFO] CatBoost予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_cat.csv


INFO:07_poc_feature_engineering_for_models:CatBoost予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_cat.csv


In [19]:
print("=" * 60)
print(f"{datetime.datetime.now()} - HistGradientBoosting v2 実行開始")
print("=" * 60)

from common.histgb.histgb_model_v2 import run_histgb

histgb_params = base_params.copy()

# 入社日をTimeSeriesSplit用に追加
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

data = {"X_train": X_train_with_sort, "y_train": y_train, "X_test": X_test, "sort_col": "入社日"}
result_data, _ = run_histgb(data, histgb_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nHistGradientBoosting v2 OOF Score (Log Loss): {oof_score:.6f}")
results["HistGradientBoosting"] = oof_score

submission_histgb = pd.DataFrame({ID_COL: test_persona[ID_COL], TARGET_COL: test_preds})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_histgb.csv"
submission_histgb.to_csv(submission_path, index=False,header=False)
logger.info(f"HistGradientBoosting予測結果保存: {submission_path}")

2026-08-06 16:41:49.095465 - HistGradientBoosting v2 実行開始

HistGradientBoosting v2 OOF Score (Log Loss): 0.696567
[2026-08-06 16:41:53] [INFO] HistGradientBoosting予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_histgb.csv


INFO:07_poc_feature_engineering_for_models:HistGradientBoosting予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_histgb.csv


In [20]:
print("=" * 60)
print(f"{datetime.datetime.now()} - Logistic Regression v2 実行開始")
print("=" * 60)

from common.logistic.logistic_model_v2 import run_logistic

logistic_params = base_params.copy()
logistic_params.update({"max_iter": 1000})

# 入社日をTimeSeriesSplit用に追加
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

data = {"X_train": X_train_with_sort, "y_train": y_train, "X_test": X_test, "sort_col": "入社日"}
result_data, _ = run_logistic(data, logistic_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nLogistic Regression v2 OOF Score (Log Loss): {oof_score:.6f}")
results["Logistic Regression"] = oof_score

submission_logistic = pd.DataFrame({ID_COL: test_persona[ID_COL], TARGET_COL: test_preds})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_logistic.csv"
submission_logistic.to_csv(submission_path, index=False,header=False)
logger.info(f"Logistic Regression予測結果保存: {submission_path}")

2026-08-06 16:41:53.390483 - Logistic Regression v2 実行開始

Logistic Regression v2 OOF Score (Log Loss): 0.816139
[2026-08-06 16:41:58] [INFO] Logistic Regression予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_logistic.csv


INFO:07_poc_feature_engineering_for_models:Logistic Regression予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_logistic.csv


In [21]:
print("=" * 60)
print(f"{datetime.datetime.now()} - Neural Network v2 実行開始")
print("=" * 60)

from common.nn.nn_model_v2 import run_nn

nn_params = base_params.copy()
nn_params.update({
    "hidden_dim": 128,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 50,
})

# 入社日をTimeSeriesSplit用に追加し、数値化（PyTorchはobject型を扱えないため）
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
# 日付を数値（unix timestamp）に変換
X_train_with_sort['入社日'] = (train_features_with_date['入社日'] - pd.Timestamp('1970-01-01')).dt.total_seconds()

data = {"X_train": X_train_with_sort, "y_train": y_train, "X_test": X_test, "sort_col": "入社日"}
result_data, _ = run_nn(data, nn_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nNeural Network v2 OOF Score (Log Loss): {oof_score:.6f}")
results["Neural Network"] = oof_score

submission_nn = pd.DataFrame({ID_COL: test_persona[ID_COL], TARGET_COL: test_preds})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_nn.csv"
submission_nn.to_csv(submission_path, index=False,header=False)
logger.info(f"Neural Network予測結果保存: {submission_path}")

2026-08-06 16:41:58.827388 - Neural Network v2 実行開始

Neural Network v2 OOF Score (Log Loss): 1.119381
[2026-08-06 16:42:25] [INFO] Neural Network予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_nn.csv


INFO:07_poc_feature_engineering_for_models:Neural Network予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_nn.csv


In [29]:
import datetime
print("=" * 60)
print(f"{datetime.datetime.now()} - TabNet v2 実行開始")
print("=" * 60)

from common.tabnet.tabnet_model_v2 import run_tabnet

# TabNet用にカテゴリカル変数の名称を指定
cat_feature_names = ["入社区分", "性別", "最終学歴", "専攻分野", "採用経路", "初期職種", "初期勤務地", "初期等級", "初期役割"]

tabnet_params = base_params.copy()
tabnet_params.update({
    "n_d": 32,
    "n_a": 32,
    "n_steps": 5,
    "gamma": 1.5,
    "lambda_sparse": 1e-4,
    "cat_feature_names": cat_feature_names,
    "cat_emb_dim": 3,  # カテゴリ埋め込み次元（表現力を高めるために拡張）
})

# 入社日をTimeSeriesSplit用に追加
train_features_with_date = train_persona[[ID_COL, '入社日']].copy()
X_train_with_sort = X_train.copy()
# 日付を数値（unix timestamp）に変換
X_train_with_sort['入社日'] = (train_features_with_date['入社日'] - pd.Timestamp('1970-01-01')).dt.total_seconds()

X_test_with_sort = X_test.copy()
test_features_with_date = test_persona[[ID_COL, '入社日']].copy()
X_test_with_sort['入社日'] = (test_features_with_date['入社日'] - pd.Timestamp('1970-01-01')).dt.total_seconds()

X_train_tabnet = X_train_with_sort.copy()
X_test_tabnet = X_test_with_sort.copy()

# 一旦 -999 を NaN に戻す
X_train_tabnet = X_train_tabnet.replace(-999, np.nan).replace(-999.0, np.nan)
X_test_tabnet = X_test_tabnet.replace(-999, np.nan).replace(-999.0, np.nan)

# 全てがNaNのカラムを削除（これがSimpleImputerでカラム数が減る原因）
all_nan_cols = X_train_tabnet.columns[X_train_tabnet.isna().all()].tolist()
if all_nan_cols:
    X_train_tabnet.drop(columns=all_nan_cols, inplace=True)
    X_test_tabnet.drop(columns=all_nan_cols, inplace=True)
    # cat_feature_namesからも削除
    cat_feature_names = [c for c in cat_feature_names if c not in all_nan_cols]
    tabnet_params["cat_feature_names"] = cat_feature_names

# 事前に中央値で欠損値を埋めておく（TabNet内部のImputerがカラムを削除しないようにする）
for col in X_train_tabnet.columns:
    if col not in cat_feature_names and col != '入社日':
        # 中央値計算
        med = X_train_tabnet[col].median()
        if pd.isna(med):
            med = 0.0
        X_train_tabnet[col] = X_train_tabnet[col].fillna(med).astype(float)
        X_test_tabnet[col] = X_test_tabnet[col].fillna(med).astype(float)
    elif col in cat_feature_names:
        # カテゴリカル変数の欠損は最頻値で埋める
        mode_val = X_train_tabnet[col].mode()
        fill_val = mode_val[0] if not mode_val.empty else 0
        X_train_tabnet[col] = X_train_tabnet[col].fillna(fill_val).astype(int)
        X_test_tabnet[col] = X_test_tabnet[col].fillna(fill_val).astype(int)

# '入社日' カラムのNaNも埋める
X_train_tabnet['入社日'] = X_train_tabnet['入社日'].fillna(0).astype(float)
X_test_tabnet['入社日'] = X_test_tabnet['入社日'].fillna(0).astype(float)

data = {"X_train": X_train_tabnet, "y_train": y_train, "X_test": X_test_tabnet, "sort_col": "入社日"}
result_data, _ = run_tabnet(data, tabnet_params)
test_preds = result_data["test_preds"]
oof_score = result_data["oof_score"]

print(f"\nTabNet v2 OOF Score (Log Loss): {oof_score:.6f}")
results["TabNet"] = oof_score

submission_tabnet = pd.DataFrame({ID_COL: test_persona[ID_COL], TARGET_COL: test_preds})
submission_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_tabnet.csv"
submission_tabnet.to_csv(submission_path, index=False,header=False)
logger.info(f"TabNet予測結果保存: {submission_path}")

2026-08-06 16:54:07.958754 - TabNet v2 実行開始
epoch 0  | loss: 1.45275 | val_0_logloss: 1.43118 |  0:00:00s
epoch 1  | loss: 1.25592 | val_0_logloss: 1.17307 |  0:00:00s
epoch 2  | loss: 1.00459 | val_0_logloss: 1.06285 |  0:00:00s
epoch 3  | loss: 1.00411 | val_0_logloss: 0.92659 |  0:00:00s
epoch 4  | loss: 1.0992  | val_0_logloss: 0.99199 |  0:00:00s
epoch 5  | loss: 0.87677 | val_0_logloss: 1.05348 |  0:00:00s
epoch 6  | loss: 0.87359 | val_0_logloss: 0.92631 |  0:00:00s
epoch 7  | loss: 0.93919 | val_0_logloss: 0.85276 |  0:00:00s
epoch 8  | loss: 0.88305 | val_0_logloss: 0.77812 |  0:00:00s
epoch 9  | loss: 0.82684 | val_0_logloss: 0.76389 |  0:00:00s
epoch 10 | loss: 0.8368  | val_0_logloss: 0.77724 |  0:00:00s
epoch 11 | loss: 0.7957  | val_0_logloss: 0.81517 |  0:00:01s
epoch 12 | loss: 0.80692 | val_0_logloss: 0.82374 |  0:00:01s
epoch 13 | loss: 0.82906 | val_0_logloss: 0.76915 |  0:00:01s
epoch 14 | loss: 0.71287 | val_0_logloss: 0.74009 |  0:00:01s
epoch 15 | loss: 0.71921 |

INFO:07_poc_feature_engineering_for_models:TabNet予測結果保存: /content/drive/MyDrive/jaggle_2026/data/output/20260806/20260806_07_poc_feature_engineering_for_models_tabnet.csv


## 6. 結果の比較

In [30]:
# Neural Networkのスコアを更新
results["Neural Network"] = 1.123226

print("\n" + "=" * 60)
print("特徴量エンジニアリング × 全モデル検証結果")
print("=" * 60)

results_df = pd.DataFrame([
    {"Model": model, "OOF Score (Log Loss)": score}
    for model, score in results.items()
]).sort_values("OOF Score (Log Loss)")

print("\nモデルスコア一覧（Log Loss昇順）:")
print(results_df.to_string(index=False))

print("\n" + "=" * 60)
print(f"最高スコア: {results_df.iloc[0]['Model']} - {results_df.iloc[0]['OOF Score (Log Loss)']:.6f}")
print("=" * 60)

logger.info("全モデルの実行が完了しました！")
logger.info(f"最高スコア: {results_df.iloc[0]['Model']} - {results_df.iloc[0]['OOF Score (Log Loss)']:.6f}")


特徴量エンジニアリング × 全モデル検証結果

モデルスコア一覧（Log Loss昇順）:
               Model  OOF Score (Log Loss)
            CatBoost              0.586837
            LightGBM              0.595333
             XGBoost              0.626103
              TabNet              0.660840
HistGradientBoosting              0.696567
 Logistic Regression              0.816139
      Neural Network              1.123226

最高スコア: CatBoost - 0.586837
[2026-08-06 16:55:35] [INFO] 全モデルの実行が完了しました！


INFO:07_poc_feature_engineering_for_models:全モデルの実行が完了しました！


[2026-08-06 16:55:35] [INFO] 最高スコア: CatBoost - 0.586837


INFO:07_poc_feature_engineering_for_models:最高スコア: CatBoost - 0.586837


## 結論

このノートブックでは、04_baseline_lgbm_feature_engineeringで実装した特徴量エンジニアリングを全モデルで検証しました。

**実装した特徴量:**
1. 月次データの集約特徴量（統計量、時期別、トレンド）
2. 月次カテゴリカル変数の変化パターン
3. 欠損値を活用した特徴量
4. ドメイン知識ベース特徴量（エンゲージメント、WLB、成長、コミュニケーション、ストレス）
5. 時間的特徴量
6. 交互作用特徴量

**CV戦略:** TimeSeriesSplit (cv_strategy="timeseries")

**次のステップ:**
- 最高スコアのモデルを選定
- そのモデルでOptunaハイパーパラメータ最適化を実施
- 特徴量重要度分析
- アンサンブルモデルの検討